In [ ]:
import warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from pint.errors import UnitStrippedWarning

import prism
from imagematerials.read_mym import read_mym_df

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
    SharesInflowStocks
)
from imagematerials.vehicles.battery import ElectricVehicleBatteries, BatteryMaterials, EvBatteryLinkModule, ElectricVehicleBatteriesWeights
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.electricity.preprocessing import get_preprocessing_data_evbattery, get_preprocessing_data_evbattery_old

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_data = Path(path_base, "data", "raw")

# Old

In [ ]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_data, "image", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)

    vhc_sector = get_preprocessing_data("vehicles", path_data, 
                                    climate_policy_scenario_dir = climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
    

    factory = ModelFactory(
        [vhc_sector], complete_timeline
        ).add(GenericStocks, "vehicles"
        ).add(BatteryMaterials, "vehicles"
        ).add(GenericMaterials, "vehicles"
        )
    model = factory.finish()

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UnitStrippedWarning)
        model.simulate(simulation_timeline)
    all_output[scen_id] = model
    print(f"Finished {scen_id}")

list(model.vehicles)

# New

In [ ]:
# Only batteries #

scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output_2 = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_data, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    vhc_sector = get_preprocessing_data("vehicles", path_data, 
                                    climate_policy_scenario_dir = climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
    ev_battery_sector = get_preprocessing_data("ev_battery", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario)     
    # elc_sector = get_preprocessing_data("electricity", path_data,
    #                                     climate_policy_scenario_dir, 
    #                                     circular_economy_scenario_dir, cache = False, standard_scenario = scenario) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    

    factory = ModelFactory(
        [vhc_sector, ev_battery_sector], complete_timeline
        ).add(GenericStocks, ["vehicles"]
        ).add(ElectricVehicleBatteries, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
        ).add(GenericMaterials, "vehicles"
        )
    model_2 = factory.finish()

    model_2.simulate(simulation_timeline)
    all_output_2[scen_id] = model_2
    print(f"Finished {scen_id}")

list(model_2.ev_battery)

In [ ]:
# All 3 sectors #

scenario_list = {
                "SSP2_VLHO":("SSP2_VLHO", None),
                "SSP2_M_CP":("SSP2_M_CP", None),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_data, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    vhc_sector = get_preprocessing_data("vehicles", path_data, 
                                    climate_policy_scenario_dir = Path(climate_policy_scenario_dir), 
                                    circular_economy_scenario_dirs = None)
    ev_battery_sector = get_preprocessing_data("ev_battery", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario)     
    elc_sector = get_preprocessing_data("electricity", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    

    factory = ModelFactory(
        [vhc_sector, ev_battery_sector, *elc_sector], complete_timeline
        ).add(GenericStocks, ["vehicles",
                              "elc_gen",
                              "elc_grid_lines",
                              "elc_grid_add",
                              "elc_stor_phs"]
        ).add(GenericMaterials, ["vehicles"]
        ).add(ElectricVehicleBatteriesWeights, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
        ).add(EvBatteryLinkModule, ["elc_stor_other"], input_sources={
        "stock_battery_kWh_v2g": "ev_battery"}
        ).add(SharesInflowStocks, ["elc_stor_other"],
        ).add(MaterialIntensities, ["elc_gen",
                                    "elc_grid_lines",
                                    "elc_grid_add",
                                    "elc_stor_phs",
                                    "elc_stor_other"
                                    ]
        )
    model = factory.finish()

    model.simulate(simulation_timeline)
    all_output[scen_id] = model
    print(f"Finished {scen_id}")
    del factory

list(model.ev_battery)

# Plots

In [ ]:
t = model.elc_stor_phs["stocks"].pint.to("GWh")
t

## external data

In [ ]:
# read the external files
df_ev_battery_i =       pd.read_csv(Path(path_data,"electricity","lit_ev_battery_inflow.csv"), index_col = [0])
df_ev_battery_s =       pd.read_csv(Path(path_data,"electricity","lit_ev_battery_stock.csv"), index_col = [0])
df_storage_power_i =    pd.read_csv(Path(path_data,"electricity","lit_grid_storage_inflow_power_GW.csv"), index_col = [0])
df_storage_power_s =    pd.read_csv(Path(path_data,"electricity","lit_grid_storage_stock_power_GW.csv"), index_col = [0])
df_storage_energy_i =   pd.read_csv(Path(path_data,"electricity","lit_grid_storage_inflow_energy_GWh.csv"), index_col = [0])
df_storage_energy_s =   pd.read_csv(Path(path_data,"electricity","lit_grid_storage_stock_energy_GWh.csv"), index_col = [0])
# Assumptions to convert GW to GWh
duration_battery = 4 # assume 4 h duration
duration_phd = 8 # 8 h

# TIMER
path_image = Path(path_data, "image", "SSP2_M_CP", "EnergyServices")
df_energy_timer_data = read_mym_df(Path(path_data, "image", "SSP2_M_CP", "EnergyServices", "StorResTot.out"))
df_power_timer_data = read_mym_df(Path(path_data, "image", "SSP2_M_CP", "EnergyServices", "StorCapTot.out"))
df_energy_timer_data_seb = read_mym_df(Path(path_data, "image", "SSP2_BL", "StorResTot.out"))
df_power_timer_data_seb = read_mym_df(Path(path_data, "image", "SSP2_BL",  "StorCapTot.out"))

df_energy_timer = df_energy_timer_data.rename(columns={28: "global"})/1000 # MWh->GWh
df_energy_timer_seb = df_energy_timer_data_seb.rename(columns={28: "global"})/1000 # MWh->GWh

In [ ]:
# Plot IMAGE storage #

scen = "SSP2_M_CP"
model = all_output[scen]

fig, ax = plt.subplots(figsize=(8,6))
s_0 = model.elc_stor_other["stocks_non_phs"].sum(["SuperType", "Region"]).pint.to("GWh")
s = model.elc_stor_other["stocks"].sum(["SuperType", "Region"]).pint.to("GWh")
phs = model.elc_stor_phs["stocks"].sum(["Type", "Region"]).pint.to("GWh")
phs_stacked = phs + s_0

# plt.plot(phs.Time, phs, label="PHS")
plt.plot(phs_stacked.Time, phs_stacked, label="dedicated storage + ev + PHS")
plt.plot(s_0.Time, s_0, label="dedicated storage + ev batteries (v2g)")
plt.plot(s.Time, s, label="dedicated storage")
ax.plot(df_energy_timer.index, df_energy_timer["global"], linewidth=1.5, c="black",
        label="TIMER - total storage demand", linestyle="--")

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"Storage stocks (GWh)", fontsize=14)
plt.legend()
plt.title(f"{scen}: Stocks of storage technologies", fontsize=14)

In [ ]:
# INFLOW EV BATTERIES GWh #

df = df_ev_battery_i.loc[(df_ev_battery_i["Technology"] == "all_modes")&(df_ev_battery_i["unit"] == "GWh")]

# Plot ev_battery: materials over time
var = "inflow_battery_kWh"
model = all_output["SSP2_M_CP"]
m2 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None))/1000000
model = all_output["SSP2_VLHO"]
m1 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None))/1000000

fig, ax = plt.subplots(figsize=(8,6))
plt.plot(m1.time, m1, label="IMAGE-Materials - SSP2_VLHO", linewidth=1, c="black", linestyle = "--")
plt.plot(m2.time, m2, label="IMAGE-Materials - SSP2_M_CP", linewidth=1, c="black")
for (source, scenario), g in df.groupby(["source", "scenario"]):
    plt.scatter(g.index, g["value"], label=f"{source} – {scenario}", linewidth=1)

# plt.xlim(2010,2060)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"Inflow EV batteries (GWh)", fontsize=14)
plt.legend()
plt.title(f"Inflow EV batteries", fontsize=14)

In [ ]:
model = all_output["SSP2_M_CP"]
m = model.ev_battery["energy_density"].sum(["BatteryType"])
m
fig, ax = plt.subplots(figsize=(8,6))
plt.plot(m.Cohort, m, label="IMAGE-Materials - SSP2_M_CP", linewidth=1, c="black")

In [ ]:
list(all_output["SSP2_M_CP"].ev_battery)

In [ ]:
var = "stock_battery_kg"
model = all_output["SSP2_M_CP"]
m2 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type", "Cohort"]).sel(time=slice(1972,None))
m2

In [ ]:
# STOCK EV BATTERIES kg #

df = df_ev_battery_s.loc[(df_ev_battery_s["Technology"] == "all_modes")&(df_ev_battery_s["unit"] == "kg")]

# Plot ev_battery: materials over time
model = all_output["SSP2_M_CP"]
m4 = model.vehicles["stock_by_cohort_materials"].sum(["material", "Region", "Type"]).sel(Time=slice(1972,None))
m3 = model.elc_stor_other["stock_by_cohort_materials"].sum(["material", "Region", "Type"]).sel(Time=slice(1972,None))
var = "stock_battery_kg"
m2 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type", "Cohort"]).sel(time=slice(1972,None))
m21 = model.ev_battery["stock_battery_materials"].to_array().sum(["BatteryType", "Region", "Type", "material"]).sel(time=slice(1972,None))
model = all_output["SSP2_VLHO"]
m1 = model.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type", "Cohort"]).sel(time=slice(1972,None))

fig, ax = plt.subplots(figsize=(8,6))
plt.plot(m4.Time, m4, label="IMAGE-Materials - vehicles", linewidth=1, c="purple", linestyle = ":")
plt.plot(m3.Time, m3, label="IMAGE-Materials - dedicated storage", linewidth=1, c="black", linestyle = ":")
# plt.plot(m1.time, m1, label="IMAGE-Materials - SSP2_VLHO", linewidth=1, c="black", linestyle = "--")
plt.plot(m2.time, m2, label="IMAGE-Materials - SSP2_M_CP", linewidth=1, c="black")
plt.plot(m21.time, m21, label="IMAGE-Materials - sum materials", linewidth=1, c="orange")
for (source, scenario), g in df.groupby(["source", "scenario"]):
    plt.scatter(g.index, g["value"], label=f"{source} – {scenario}", linewidth=1)

# plt.xlim(2010,2060)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"Stock EV batteries (kg)", fontsize=14)
plt.legend()
plt.title(f"Stock EV batteries", fontsize=14)

In [ ]:
# TIMER comparison
# comparison of SSP2_BL which Sebastiaan used to newer SSP2_M_CP runs

df = df_storage_energy_s.copy()
df = df[(df["source"] == "IRENA") & (df["category"] == "total")]
df.reset_index(drop=False,inplace=True)

fig, ax = plt.subplots(figsize=(8,6))

# first y-axis: total storage
ax.plot(df_energy_timer.index, df_energy_timer["global"], linewidth=2, c="darkgray",
        label="TIMER_SSP2_M_CP - total storage")
ax.plot(df_energy_timer_seb.index, df_energy_timer_seb["global"], linewidth=2, c="black",
        label="TIMER SSP2_BL - total storage")
for row in range(len(df)):
    src = df["source"].iloc[row]
    scen = df["scenario"].iloc[row]
    plt.scatter(df["time"].iloc[row], df["value"].iloc[row], label=f"{src} – {scen}", linewidth=1)

ax.set_xlabel("Year", fontsize=14)
ax.set_ylabel("Storage stocks (GWh)", fontsize=14)
ax.tick_params(axis='both', labelsize=14)

# second y-axis: ratio
ax2 = ax.twinx()
ratio = df_energy_timer["global"] / df_energy_timer_seb["global"]
ax2.plot(df_energy_timer.index, ratio, linestyle="--", c="rosybrown", alpha=0.65, label="Ratio (M_CP / BL)")
ax2.set_ylabel("Ratio (-)", fontsize=14)
ax2.tick_params(axis='y', labelsize=14)

# legend (combined)
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines + lines2, labels + labels2)

ax.set_title("Global energy storage (stocks)", fontsize=14)


In [ ]:
df = df_storage_power_s
df_sel = df[df["sub_technology"].isna() & (df["technology"] == "battery")]
df_sel.reset_index(drop=False,inplace=True)
group_cols = ["time", "source", "scenario", "type_data", "unit", "region", "technology", "sub_technology",]

df_sum = (df_sel.groupby(group_cols, dropna=False, as_index=False)["value"].sum())
df_sum

In [ ]:
df = df_storage_energy_s
df_sel = df[(df["category"] == "grid_scale_storage") & (df["technology"] == "battery")]
df_sel

list(model.elc_stor_phs)
model.elc_stor_phs["stocks"]

In [ ]:
# Plot grid storage #

# EXTERNAL DATA
df = df_storage_power_s
df_sel = df[df["sub_technology"].isna() & (df["technology"] == "battery")]
df_sel.reset_index(drop=False,inplace=True)
group_cols = ["time", "source", "scenario", "type_data", "unit", "region", "technology", "sub_technology",]
df_sum = (df_sel.groupby(group_cols, dropna=False, as_index=False)["value"].sum())
df_sum["value"] = df_sum["value"] * duration_battery

df = df_storage_energy_s
df_energy = df[(df["category"] == "grid_scale_storage") & (df["technology"] == "battery")]
df_energy.reset_index(drop=False,inplace=True)

# df2 = df_storage_energy_s.copy()
# df2 = df2[(df2["source"] == "IRENA") & (df2["category"] == "total")]
# df2.reset_index(drop=False,inplace=True)

# PLOT
fig, ax = plt.subplots(figsize=(8,6))
model = all_output["SSP2_M_CP"]
s_0 = model.elc_stor_other["stocks_non_phs"].sum(["SuperType", "Region"]).pint.to("GWh")
s = model.elc_stor_other["stocks"].sum(["SuperType", "Region"]).pint.to("GWh")
test = s+s_0
phs = model.elc_stor_phs["stocks"].sum(["Type", "Region"]).pint.to("GWh")

plt.plot(phs.Time, phs, label="PHS")
plt.plot(s_0.Time, s_0, label="dedicated storage + ev batteries (v2g)")
plt.plot(s.Time, s, label="dedicated storage")
# plt.plot(test.Time, test, label="test", linestyle=":")
plt.plot(df_energy_timer.index, df_energy_timer["global"], label="TIMER_SSP2_M_CP - total storage", c="black")
plt.plot(df_energy_timer_seb.index, df_energy_timer_seb["global"], label="TIMER SSP2_BL - total storage", c="black", linestyle="--")
for row in range(len(df_sum)):
    src = df_sum["source"].iloc[row]
    scen = df_sum["scenario"].iloc[row]
    plt.scatter(df_sum["time"].iloc[row], df_sum["value"].iloc[row], label=f"{src} – {scen}", linewidth=1)
for row in range(len(df_energy)):
    src = df_energy["source"].iloc[row]
    scen = df_energy["scenario"].iloc[row]
    plt.scatter(df_energy["time"].iloc[row], df_energy["value"].iloc[row], label=f"{src} – {scen}", linewidth=1)
# plt.scatter(df_sum.time, df_sum.values, label="PHS")
# for row in range(len(df)):
#     src = df["source"].iloc[row]
#     scen = df["scenario"].iloc[row]
#     plt.scatter(df["time"].iloc[row], df["value"].iloc[row], label=f"{src} – {scen}", linewidth=1)
# for row in range(len(df2)):
#     src = df2["source"].iloc[row]
#     scen = df2["scenario"].iloc[row]
#     plt.scatter(df2["time"].iloc[row], df2["value"].iloc[row], label=f"{src} – {scen}", linewidth=1)

ax.set_xlim(2020,2050)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"Storage stocks (GWh)", fontsize=14)
plt.legend()
plt.title(f"Stocks of storage technologies", fontsize=14)

In [ ]:
# Plot PHS storage #

df = df_storage_energy_s.loc[df_storage_energy_s.index<2100]

fig, ax = plt.subplots(figsize=(8,6))
# s_0 = model.elc_stor_other["stocks_0"].sum(["SuperType", "Region"]).pint.to("GWh")
# s = model.elc_stor_other["stocks"].sum(["SuperType", "Region"]).pint.to("GWh")
phs = model.elc_stor_phs["stocks"].sum(["Type", "Region"]).pint.to("GWh")

plt.plot(phs.Time, phs, label="I-Materials - SSP2", c="black")
for (source, scenario), g in df.groupby(["source", "scenario"]):
    plt.scatter(g.index, g["value"], label=f"{source} – {scenario}", linewidth=1, c="darkorange")

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"PHS stocks (GWh)", fontsize=14)
plt.legend()
plt.title(f"Stocks of Pumped Hydropower Storage", fontsize=14)

## old vs. new

In [ ]:
# Plot ev_battery: materials over time
var = "inflow_battery_materials"
m1 = model.vehicles[var].sum(["battery", "Region", "Type"]).sel(Time=slice(1972,None))
m2 = model_2.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None))

for mat in m1.material.values[:4]: #
    plt.plot(m1.Time, m1.sel(material=mat), label="old - " + str(mat))
    plt.plot(m2.time, m2.sel(material=mat), label="new - " + str(mat), linestyle=":", linewidth=2)
    
plt.xlabel("Year")
plt.ylabel(f"{var} in EV batteries (kg)")
plt.legend()
plt.title(f"{var} in EV batteries")

In [ ]:
# Plot ev_battery: materials over time
var = "outflow_battery_materials"
m1 = model.vehicles[var].sum(["battery", "Region", "Type"]).sel(Time=slice(1972,None))
m2 = model_2.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None))

for mat in m1.material.values[:4]: #
    plt.plot(m1.Time, m1.sel(material=mat), label="m1 - " + str(mat))
    plt.plot(m2.time, m2.sel(material=mat), label="m2 - " + str(mat), linestyle=":", linewidth=2)
plt.xlabel("Year")
plt.ylabel(f"{var} in EV batteries (kg)")
plt.legend()
plt.title(f"{var} in EV batteries")